# Extraction de la Colonne Vertébrale (Backbone) d'un Réseau Complexe

Ce notebook contient l'implémentation complète, modulaire et commentée pour l'extraction et la visualisation du **backbone** du réseau des philosophes via le **Filtre de Disparité** (*Disparity Filter*, Serrano et al., 2009).

### Sommaire des Tâches :
1. **Analyse Force ($s$) vs Degré ($k$)** : Scatter plot en échelle log-log et annotation automatique de *Diogène Laërce* et des penseurs les plus éloignés de la diagonale.
2. **Implémentation 'from scratch' du Filtre de Disparité** : Évaluation mathématique des $p$-values et tableau comparatif avec les seuils globaux naïfs ($w \ge 2, 3, 4$).
3. **Visualisation de la figure principale ($\alpha = 0.2$)** : Application rigoureuse des 5 règles de visualisation (Louvain sur réseau non pondéré complet, layout sur le backbone, taille proportionnelle à la force, top 10 penseurs annotés, légende et caption).
4. **Comparaison visuelle ($\alpha = 0.1$ et $\alpha = 0.3$)** : Analyse de l'impact du seuil de significativité sur la structure et la densité du squelette.



In [ ]:
import os
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Configuration visuelle
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

print("Environnement prêt !")



## 1. Chargement du réseau pondéré et extraction de la GCC
- Addition des poids des arêtes dirigées ($A \to B$ et $B \to A$) pour obtenir un graphe non orienté pondéré.
- Extraction de la plus grande composante connexe (GCC).



In [ ]:
def load_philosophers_network(filepath="philosophers.edgelist", fallback_tsv="week4_philosophers_edges.tsv"):
    G = nx.Graph()
    if os.path.exists(filepath):
        print(f"Chargement depuis '{filepath}'...")
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'): continue
                parts = line.split()
                u, v = parts[0], parts[1]
                w = float(parts[2]) if len(parts) > 2 else 1.0
                if G.has_edge(u, v):
                    G[u][v]['weight'] += w
                else:
                    G.add_edge(u, v, weight=w)
    elif os.path.exists(fallback_tsv):
        print(f"Chargement depuis '{fallback_tsv}'...")
        df_edges = pd.read_csv(fallback_tsv, sep='\t', comment='#')
        for _, row in df_edges.iterrows():
            u, v, w = row['source'], row['target'], float(row['weight'])
            if G.has_edge(u, v):
                G[u][v]['weight'] += w
            else:
                G.add_edge(u, v, weight=w)
    else:
        raise FileNotFoundError("Données introuvables.")

    gcc_nodes = max(nx.connected_components(G), key=len)
    gcc = G.subgraph(gcc_nodes).copy()
    print(f"Composante géante (GCC) : {gcc.number_of_nodes()} nœuds, {gcc.number_of_edges()} arêtes.")
    return gcc

G = load_philosophers_network()



## 2. Tâche 1 : Analyse Force ($s$) vs Degré ($k$)
- Calcul du degré $k_i$ et de la force $s_i = \sum_j w_{ij}$.
- Nuage de points log-log.
- Détection et annotation automatique de **Diogène Laërce** et des autres philosophes les plus éloignés de la diagonale théorique $s = k$.



In [ ]:
# Calcul des degrés et forces
degrees = dict(G.degree())
strengths = dict(G.degree(weight='weight'))

nodes = list(G.nodes())
k_vals = np.array([degrees[n] for n in nodes], dtype=float)
s_vals = np.array([strengths[n] for n in nodes], dtype=float)

# Distance à la diagonale s = k dans l'espace log10
log_k = np.log10(k_vals)
log_s = np.log10(s_vals)
perp_dist = (log_s - log_k) / np.sqrt(2.0)
ratios = s_vals / k_vals

df_nodes = pd.DataFrame({
    'node': nodes,
    'degree': k_vals,
    'strength': s_vals,
    'ratio_s_k': ratios,
    'dist_diagonal': perp_dist
})

# Sélection des penseurs à annoter
target = "Diogenes_Laertius"
other_outliers = (
    df_nodes[df_nodes['node'] != target]
    .sort_values(by='dist_diagonal', ascending=False)
    .head(3)['node']
    .tolist()
)
annotated = [target] + other_outliers

print("Philosophes annotés :")
for n in annotated:
    row = df_nodes[df_nodes['node'] == n].iloc[0]
    print(f" - {row['node']:25s} : k = {row['degree']:3.0f}, s = {row['strength']:4.0f}, s/k = {row['ratio_s_k']:.2f}")

# Tracé graphique
fig, ax = plt.subplots(figsize=(9, 6.5))

scatter = ax.scatter(k_vals, s_vals, alpha=0.55, c=ratios, cmap='viridis', s=28, edgecolors='none', zorder=2)
cbar = plt.colorbar(scatter, ax=ax, pad=0.02)
cbar.set_label("Poids moyen par lien ($s_i / k_i$)", fontsize=10)

k_min, k_max = max(1, k_vals.min()), k_vals.max()
ax.plot([k_min, k_max], [k_min, k_max], color='#e74c3c', linestyle='--', linewidth=1.8, label="Diagonale $s = k$ (poids = 1)")

mean_w = np.sum(s_vals) / np.sum(k_vals)
ax.plot([k_min, k_max], [mean_w * k_min, mean_w * k_max], color='#2980b9', linestyle=':', linewidth=1.8,
        label=f"Moyenne globale $s = \\langle w \\rangle k$ (\\langle w \\rangle = {mean_w:.2f})")

for i, name in enumerate(annotated):
    row = df_nodes[df_nodes['node'] == name].iloc[0]
    xk, ys = row['degree'], row['strength']
    clean_name = name.replace('_', ' ')
    if name == "Diogenes_Laertius":
        xytext = (xk * 0.45, ys * 1.55)
        color_box = '#ffeaa7'
    else:
        xytext = (xk * 0.55, ys * (1.6 + 0.4 * (i % 2)))
        color_box = '#ffffff'

    ax.annotate(
        f"{clean_name}\n($k={int(xk)}, s={int(ys)}$)",
        xy=(xk, ys), xytext=xytext,
        arrowprops=dict(facecolor='black', shrink=0.08, width=0.8, headwidth=5),
        fontsize=8.5, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.25', facecolor=color_box, edgecolor='#333333', lw=0.7, alpha=0.9),
        zorder=5
    )

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel("Degré du philosophe $k_i$ (échelle log)", fontsize=11, fontweight='bold')
ax.set_ylabel("Force du philosophe $s_i$ (échelle log)", fontsize=11, fontweight='bold')
ax.set_title("Analyse Force vs Degré : Identification des Penseurs Pivot", fontsize=13, fontweight='bold')
ax.grid(True, which="both", ls="--", lw=0.5, alpha=0.5)
ax.legend(loc='lower right', frameon=True)
plt.tight_layout()
plt.show()



## 3. Tâche 2 : Implémentation 'from scratch' du Filtre de Disparité
- Formule : $p_{ij} = w_{ij} / s_i$.
- Test de significativité : $\alpha_{ij} = (1 - p_{ij})^{k_i - 1} < \alpha$.
- Rétention : conservé si significatif pour au moins l'une des extrémités.
- Comparaison avec seuils globaux naïfs ($w \ge 2, 3, 4$).



In [ ]:
def disparity_filter(G, alpha):
    """Implémentation 'from scratch' du Filtre de Disparité (Serrano et al., 2009)."""
    degrees = dict(G.degree())
    strengths = dict(G.degree(weight='weight'))
    backbone = nx.Graph()

    for u, v, data in G.edges(data=True):
        w = float(data.get('weight', 1.0))
        k_u, s_u = degrees[u], strengths[u]
        k_v, s_v = degrees[v], strengths[v]

        p_u = w / s_u
        p_v = w / s_v

        alpha_u = (1.0 - p_u) ** (k_u - 1) if k_u > 1 else 1.0
        alpha_v = (1.0 - p_v) ** (k_v - 1) if k_v > 1 else 1.0

        if alpha_u < alpha or alpha_v < alpha:
            backbone.add_edge(u, v, weight=w)

    return backbone

# Tableau comparatif
alphas = [0.05, 0.1, 0.2, 0.3, 0.5]
naive_thresholds = [2, 3, 4]
n_total_edges = G.number_of_edges()
n_total_nodes = G.number_of_nodes()

records = []
for a in alphas:
    B = disparity_filter(G, a)
    e = B.number_of_edges()
    n = B.number_of_nodes()
    gcc_sz = len(max(nx.connected_components(B), key=len)) if n > 0 else 0
    records.append({
        'Méthode': f'Filtre de Disparité (α = {a})',
        'Paramètre': f'α = {a}',
        'Liens conservés': e,
        '% Liens': f"{(e/n_total_edges)*100:.1f}%",
        'Philosophes (k ≥ 1)': n,
        '% Nœuds': f"{(n/n_total_nodes)*100:.1f}%",
        'Taille GCC': gcc_sz,
        '% GCC': f"{(gcc_sz/n_total_nodes)*100:.1f}%"
    })

for w_min in naive_thresholds:
    B_naive = nx.Graph()
    for u, v, data in G.edges(data=True):
        if data.get('weight', 1.0) >= w_min:
            B_naive.add_edge(u, v, weight=data.get('weight', 1.0))
    e = B_naive.number_of_edges()
    n = B_naive.number_of_nodes()
    gcc_sz = len(max(nx.connected_components(B_naive), key=len)) if n > 0 else 0
    records.append({
        'Méthode': f'Seuil Global Naïf (w ≥ {w_min})',
        'Paramètre': f'w ≥ {w_min}',
        'Liens conservés': e,
        '% Liens': f"{(e/n_total_edges)*100:.1f}%",
        'Philosophes (k ≥ 1)': n,
        '% Nœuds': f"{(n/n_total_nodes)*100:.1f}%",
        'Taille GCC': gcc_sz,
        '% GCC': f"{(gcc_sz/n_total_nodes)*100:.1f}%"
    })

df_comparison = pd.DataFrame(records)
display(df_comparison)



## 4. Tâche 3 & 4 : Visualisation du Backbone ($\alpha = 0.2$, $\alpha = 0.1$, $\alpha = 0.3$)
Application stricte des 5 règles :
1. Communautés de Louvain calculées sur le réseau **complet non pondéré** d'origine.
2. `spring_layout` calculé **uniquement** sur le réseau filtré du backbone.
3. Nœuds colorés selon leur communauté Louvain avec légende claire.
4. Taille des nœuds proportionnelle à leur force globale $s_i$.
5. Étiquettes de texte uniquement pour les 10 philosophes de plus forte force.
6. Légende descriptive (caption) sous la figure.



In [ ]:
# Règle 1 : Louvain sur le réseau complet NON PONDÉRÉ
G_unweighted = nx.Graph(G)
louvain_comms = sorted(nx.community.louvain_communities(G_unweighted, weight=None, seed=42), key=len, reverse=True)
node2comm = {node: cid for cid, c in enumerate(louvain_comms) for node in c}

comm_labels = {}
for cid, comm in enumerate(louvain_comms):
    top_thinkers = sorted(comm, key=lambda n: strengths.get(n, 0), reverse=True)[:2]
    names = ", ".join([t.replace('_', ' ') for t in top_thinkers])
    comm_labels[cid] = f"C{cid+1} ({names})"

def plot_backbone_network(G_full, G_backbone, communities, alpha, top_k_labels=10, save_path=None, seed=42):
    strengths_full = dict(G_full.degree(weight='weight'))
    nodes_backbone = list(G_backbone.nodes())
    if not nodes_backbone: return

    # Règle 2 : Layout uniquement sur le backbone
    pos = nx.spring_layout(G_backbone, seed=seed, k=0.14, iterations=60)
    fig, ax = plt.subplots(figsize=(13, 10), facecolor='white')

    cmap = plt.cm.get_cmap('tab10', len(communities))
    # Règle 4 : Taille proportionnelle à s_i
    node_sizes = [min(max(strengths_full.get(n, 1.0) * 1.6, 12), 650) for n in nodes_backbone]
    node_colors = [cmap(node2comm.get(n, 0) % 10) for n in nodes_backbone]

    # Arêtes et nœuds (Règle 3)
    nx.draw_networkx_edges(G_backbone, pos, ax=ax, alpha=0.20, edge_color='#555555', width=0.65)
    nx.draw_networkx_nodes(G_backbone, pos, ax=ax, nodelist=nodes_backbone, node_size=node_sizes,
                           node_color=node_colors, alpha=0.88, linewidths=0.5, edgecolors='#222222')

    # Règle 5 : Étiquettes pour les 10 plus forts
    top_nodes = sorted(nodes_backbone, key=lambda n: strengths_full.get(n, 0), reverse=True)[:top_k_labels]
    for n in top_nodes:
        x, y = pos[n]
        ax.text(x, y + 0.016, n.replace('_', ' '), fontsize=8.5, fontweight='bold', ha='center', va='bottom',
                color='#111111', bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.82, edgecolor='#555555', lw=0.6), zorder=6)

    # Légende explicite
    present_cids = sorted(list(set(node2comm[n] for n in nodes_backbone)))
    legend_patches = [mpatches.Patch(color=cmap(cid % 10), label=comm_labels[cid]) for cid in present_cids]
    ax.legend(handles=legend_patches, loc='upper left', title="Communautés Louvain (G complet non pondéré)",
              fontsize=8.5, title_fontsize=9.5, frameon=True, facecolor='#fafafa', edgecolor='#cccccc')

    n_full_edges = G_full.number_of_edges()
    n_back_edges = G_backbone.number_of_edges()
    n_removed = n_full_edges - n_back_edges
    pct_removed = (n_removed / n_full_edges) * 100

    ax.set_title(f"Backbone du Réseau des Philosophes (Filtre de Disparité, $\\alpha = {alpha}$)", fontsize=14, fontweight='bold', pad=12)
    ax.axis('off')

    # Caption descriptive sous la figure
    caption_text = (
        f"Filtre : Disparity Filter (Serrano et al., 2009) | Seuil : α = {alpha} | "
        f"Arêtes initiales : {n_full_edges:,} | Arêtes conservées : {n_back_edges:,} | "
        f"Arêtes retirées : {n_removed:,} ({pct_removed:.1f}%) | Philosophes connectés : {len(nodes_backbone):,}\n"
        f"Couleurs : communautés Louvain sur le réseau complet non pondéré. Taille des nœuds proportionnelle à la force $s_i$."
    )
    plt.figtext(0.5, 0.015, caption_text, wrap=True, horizontalalignment='center', fontsize=9, style='italic',
                color='#2c3e50', bbox=dict(boxstyle='square,pad=0.5', facecolor='#f8f9fa', edgecolor='#d6dbdf'))

    plt.subplots_adjust(bottom=0.08)
    if save_path: plt.savefig(save_path, dpi=180, bbox_inches='tight')
    plt.show()

# Figure principale : alpha = 0.2
print("Figure principale : alpha = 0.2")
B_02 = disparity_filter(G, alpha=0.2)
plot_backbone_network(G, B_02, louvain_comms, alpha=0.2)

# Comparaisons visuelles : alpha = 0.1 et alpha = 0.3
print("Comparaison : alpha = 0.1")
B_01 = disparity_filter(G, alpha=0.1)
plot_backbone_network(G, B_01, louvain_comms, alpha=0.1)

print("Comparaison : alpha = 0.3")
B_03 = disparity_filter(G, alpha=0.3)
plot_backbone_network(G, B_03, louvain_comms, alpha=0.3)

